In [ ]:
import ee
import geemap

# ========== 初始化 ==========
try:
    ee.Initialize(project='lst-475011')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='lst-475011')

# ========== 1) 研究区 ==========
country = (
    ee.FeatureCollection("FAO/GAUL/2015/level0")
    .filter(ee.Filter.eq("ADM0_NAME", "Netherlands"))
)
AOI = country.geometry()

# ========== 2) 年份范围（2020–2025） ==========
YEARS = [2022]
SEASONS = ["summer", "winter"]

# ========== 3) 云/阴影掩膜 + LST(℃) ==========
def mask_l2_clouds(img):
    qa = img.select("QA_PIXEL")
    cloud = qa.bitwiseAnd(1 << 3).neq(0)
    shadow = qa.bitwiseAnd(1 << 4).neq(0)
    mask = cloud.Or(shadow).Not()
    return img.updateMask(mask)


def to_lst_c_b10(img):
    # Landsat 8/9 L2: ST_B10 -> Kelvin scale factor/offset -> Celsius
    lst_k = img.select("ST_B10").multiply(0.00341802).add(149.0)
    lst_c = lst_k.subtract(273.15).rename("LST_C")
    return lst_c.copyProperties(img, img.propertyNames())

# ========== 4) 季节时间段 ==========
def season_date_range(year, season):
    if season == "summer":
        start = ee.Date.fromYMD(year, 6, 1)
        end = ee.Date.fromYMD(year, 8, 31).advance(1, "day")  # exclusive end
        return start, end

    if season == "winter":
        start = ee.Date.fromYMD(year - 1, 12, 1)
        end = ee.Date.fromYMD(year, 3, 1)  # exclusive end
        return start, end

    raise ValueError("season must be 'summer' or 'winter'")

# ========== 5) 仅用 Landsat 8/9 获取 LST 集合 ==========
def get_lst_collection_l89(start, end):
    col8 = (
        ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
        .filterBounds(AOI)
        .filterDate(start, end)
        .map(mask_l2_clouds)
        .map(to_lst_c_b10)
    )

    col9 = (
        ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
        .filterBounds(AOI)
        .filterDate(start, end)
        .map(mask_l2_clouds)
        .map(to_lst_c_b10)
    )

    return col8.merge(col9).select("LST_C")

# ========== 6) 按年 + 季节导出 ==========
for y in YEARS:
    for s in SEASONS:
        start, end = season_date_range(y, s)
        col = get_lst_collection_l89(start, end)

        # 防止某些冬季影像为 0 导致导出空/报错
        n = col.size().getInfo()
        if n == 0:
            print(f"⚠️ 无影像：{y} {s}（跳过）")
            continue

        lst_median = col.median().clip(AOI)

        desc = f"NL_LST_{y}_{s}_median_30m_L89"

        task = ee.batch.Export.image.toDrive(
            image=lst_median,
            description=desc,
            folder="GEE_LST",
            fileNamePrefix=desc,
            region=AOI,
            scale=30,
            crs="EPSG:3857",
            maxPixels=1e13,
        )

        task.start()
        print(f"✅ 已提交导出任务：{desc}（影像数：{n}）")

print("🎯 全部任务已提交，请到 GEE Tasks 或 Google Drive/GEE_LST 查看。")

✅ 已提交导出任务：NL_LST_2020_summer_median_30m_L89（影像数：39）
✅ 已提交导出任务：NL_LST_2020_winter_median_30m_L89（影像数：19）
✅ 已提交导出任务：NL_LST_2021_summer_median_30m_L89（影像数：40）
✅ 已提交导出任务：NL_LST_2021_winter_median_30m_L89（影像数：14）
✅ 已提交导出任务：NL_LST_2022_summer_median_30m_L89（影像数：94）
✅ 已提交导出任务：NL_LST_2022_winter_median_30m_L89（影像数：36）
✅ 已提交导出任务：NL_LST_2023_summer_median_30m_L89（影像数：80）
✅ 已提交导出任务：NL_LST_2023_winter_median_30m_L89（影像数：41）
✅ 已提交导出任务：NL_LST_2024_summer_median_30m_L89（影像数：85）
✅ 已提交导出任务：NL_LST_2024_winter_median_30m_L89（影像数：33）
✅ 已提交导出任务：NL_LST_2025_summer_median_30m_L89（影像数：86）
✅ 已提交导出任务：NL_LST_2025_winter_median_30m_L89（影像数：39）
🎯 全部任务已提交，请到 GEE Tasks 或 Google Drive/GEE_LST 查看。
